# Starter Kit FahMai RAG

This notebook walks through building a minimalist **Retrieval-Augmented Generation (RAG)** system for the FahMai electronics store Q&A challenge.

**Sections:**
- **Section 0:** Setup & LLM Test
- **Section 1:** Dense Retrieval (MiniLM)
- **Section 2:** Sparse Retrieval (BM25)
- **Section 3:** Hybrid Retrieval (RRF)
- **Section 4:** What's Next?

In [ ]:
# Load the data from kaggle and upload here.
!unzip -o data.zip

Archive:  data.zip
  inflating: data/knowledge_base/policies/cancellation_policy.md  
  inflating: data/knowledge_base/policies/membership_points_policy.md  
  inflating: data/knowledge_base/policies/return_policy.md  
  inflating: data/knowledge_base/policies/shipping_policy.md  
  inflating: data/knowledge_base/policies/warranty_policy.md  
  inflating: data/knowledge_base/products/AW-MN-001_arcwave_proview_27_4k.md  
  inflating: data/knowledge_base/products/AW-SK-001_arcwave_soundpillar_300.md  
  inflating: data/knowledge_base/products/DN-DT-001_daonuea_tower_x10.md  
  inflating: data/knowledge_base/products/DN-DT-002_daonuea_tower_x10_max.md  
  inflating: data/knowledge_base/products/DN-DT-003_daonuea_mini_pc_m1.md  
  inflating: data/knowledge_base/products/DN-DT-004_daonuea_all_in_one_27.md  
  inflating: data/knowledge_base/products/DN-DT-005_daonuea_all_in_one_24.md  
  inflating: data/knowledge_base/products/DN-LT-001_daonuea_airbook_15.md  
  inflating: data/knowledge_bas

In [ ]:
# === CONFIGURATION ===
# Change N_QUESTIONS to 100 for a full competition run.

N_QUESTIONS = 100  # 10 for demo, 100 for real submission
DATA_DIR = "/content/data"
KB_DIR = f"{DATA_DIR}/knowledge_base"

---
## Section 0: Setup & LLM Test

First, install dependencies and test the ThaiLLM API — **without** any retrieval. This shows why RAG is needed.

ThaiLLM website: https://playground.thaillm.or.th/chat/

In [ ]:
!pip install -q sentence-transformers pythainlp rank-bm25 requests python-dotenv

In [ ]:
import os, csv, re, time, requests
import numpy as np
from pathlib import Path
from google.colab import userdata

THAILLM_API_KEY = userdata.get('ThaiLLM')

In [ ]:
def ask_llm(messages, model="kbtg", max_retries=5):
    """Call ThaiLLM API with retry and rate-limit handling.

    Available models: typhoon, openthaigpt, pathumma, kbtg
    """
    url = f"http://thaillm.or.th/api/{model}/v1/chat/completions"
    headers = {"Content-Type": "application/json", "apikey": THAILLM_API_KEY}
    payload = {
        "model": "/model",
        "messages": messages,
        "max_tokens": 2024,
        "temperature": 0,
    }

    for attempt in range(max_retries):
        try:
            resp = requests.post(url, headers=headers, json=payload, timeout=120)

            # Rate limit — wait and retry
            if resp.status_code == 429:
                wait = min(2 ** attempt, 30)
                print(f"  Rate limited, waiting {wait}s...")
                time.sleep(wait)
                continue

            resp.raise_for_status()
            return resp.json()["choices"][0]["message"]["content"].strip()

        except requests.exceptions.RequestException as e:
            wait = 2 ** attempt
            print(f"  Error: {e}, retrying in {wait}s...")
            time.sleep(wait)

    return None


def parse_answer(text):
    """Extract answer number from LLM response."""
    if text is None:
        return None
    # Remove any <think>...</think> blocks (some models do chain-of-thought)
    clean = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    # Look for ANSWER: X pattern
    m = re.search(r"ANSWER:\s*(\d+)", clean)
    if m:
        return int(m.group(1))
    # Fallback: first standalone number 1-10
    for d in re.findall(r"\b(\d{1,2})\b", clean):
        if 1 <= int(d) <= 10:
            return int(d)
    return None

### Test the LLM without RAG

Let's ask a FahMai-specific question *without* any context. The LLM shouldn't know the answer.

In [ ]:
# Ask without context — LLM has no idea about FahMai's products
response = ask_llm([
    {"role": "user", "content": "Watch S3 Ultra กันน้ำได้กี่ ATM ครับ?"}
])
print("LLM response (no context):", response)
print("\n→ The LLM doesn't know FahMai-specific facts. We need RAG!")

LLM response (no context): <think>
Okay, the user is asking about the water resistance rating of the Samsung Galaxy S3 Ultra, specifically how many ATM it can withstand. First, I need to recall the specifications of the S3 Ultra. I remember that the S3 Ultra is a variant of the Galaxy S3, but I'm not entirely sure about its water resistance. 

Wait, the original Galaxy S3 was not water-resistant. I think the S3 Ultra might have some water resistance, but I need to confirm. Let me check. The S3 Ultra was released in 2013, and at that time, water resistance wasn't a common feature in smartphones. However, some models might have had IP ratings. 

Looking up the specifications, the S3 Ultra has an IP67 rating. IP67 means it's dust-tight and can withstand immersion in water up to 1 meter for 30 minutes. To convert that to ATM, I need to know that 1 meter of water is approximately 1 ATM. So, 1 meter is 1 ATM, and since it's 30 minutes, it's not a continuous pressure. 

But the user is asking

### Load Questions

In [ ]:
questions = []
with open(f"{DATA_DIR}/questions.csv", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        choices = {str(i): row[f"choice_{i}"] for i in range(1, 11)}
        questions.append({"id": int(row["id"]), "question": row["question"], "choices": choices})

print(f"Loaded {len(questions)} questions, using first {N_QUESTIONS} for this run")
print(f"\nExample — Q1: {questions[99]['question']}")
for k, v in questions[99]["choices"].items():
    print(f"  {k}. {v}")

Loaded 100 questions, using first 100 for this run

Example — Q1: สตอร์มบุ๊ก G5 ราคา ฿27,990 ซื้อมาได้ 5 วัน ยังไม่แกะกล่อง อยากคืนสินค้าได้ไหมครับ
  1. คืนได้ครับ เพราะอยู่ภายใน 15 วัน และยังไม่แกะกล่อง ตรงตามเงื่อนไขนโยบายคืนสินค้าของฟ้าใหม่ทุกข้อ ทั้งสภาพสมบูรณ์ บรรจุภัณฑ์ครบ และภายในกรอบเวลา จ่ายค่าส่งคืน ฿50-฿150
  2. คืนได้ แต่ต้องไปคืนที่สาขาฟ้าใหม่เท่านั้นเพราะเป็นสินค้าราคาเกิน ฿25,000
  3. ไม่ได้ครับ สตอร์มบุ๊ก G5 ราคา ฿27,990 เป็นสินค้า Clearance ลดล้างสต็อก ไม่รับคืนไม่ว่ากรณีใด
  4. คืนได้ แต่หัก 5% ค่าดำเนินการเพราะเป็นสินค้าโปรโมชัน
  5. คืนได้ภายใน 7 วันเท่านั้น เพราะเป็นสินค้าช่วง Mega Sale
  6. ไม่ได้เพราะเลย 3 วันแล้ว สินค้าลดราคาพิเศษมีเงื่อนไขคืนได้ภายใน 3 วันเท่านั้น ต้องรีบตัดสินใจให้เร็วกว่านี้ กรุณาตรวจสอบสเปคให้ดีก่อนซื้อทุกครั้ง
  7. คืนได้ แต่ได้คืนเป็น FahMai Wallet เครดิตเท่านั้น ไม่คืนเงินสด
  8. ไม่ได้ เพราะ StormBook G5 เป็นสินค้า Pre-order ที่จัดส่งแล้ว ต้องใช้นโยบายคืนสินค้าไม่ใช่ยกเลิก
  9. ไม่มีข้อมูลเพียงพอในฐานข้อมูลสำหรับตอบคำถามนี้
  10. คำถามนี

---
## Section 1: Dense Retrieval (MiniLM)

**Dense retrieval** converts text into vectors (embeddings) and finds relevant chunks by cosine similarity.

The pipeline: **Load docs → Chunk → Embed → Retrieve → Generate**

### 1.1 Load Knowledge Base

In [ ]:
kb_dir = Path(KB_DIR)
documents = []
for fp in sorted(kb_dir.rglob("*.md")):
    text = fp.read_text(encoding="utf-8")
    documents.append({"path": str(fp.relative_to(kb_dir)), "text": text})

print(f"Loaded {len(documents)} documents")
print(f"  products/: {sum(1 for d in documents if 'products/' in d['path'])}")
print(f"  policies/: {sum(1 for d in documents if 'policies/' in d['path'])}")
print(f"  store_info/: {sum(1 for d in documents if 'store_info/' in d['path'])}")

# Preview one document
print(f"\n--- Sample: {documents[0]['path']} ---")
print(documents[0]["text"][:500])

Loaded 118 documents
  products/: 110
  policies/: 5
  store_info/: 3

--- Sample: policies/cancellation_policy.md ---
# นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่

**วันที่อัปเดต:** 1 มีนาคม 2569

---

## 1. ภาพรวมนโยบาย

ฟ้าใหม่เข้าใจว่าบางครั้งลูกค้าอาจต้องการยกเลิกคำสั่งซื้อด้วยเหตุผลต่างๆ นโยบายนี้อธิบายสิทธิ์และขั้นตอนการยกเลิกคำสั่งซื้อตามสถานะของคำสั่งซื้อในขณะนั้น ความสามารถในการยกเลิกขึ้นอยู่กับสถานะคำสั่งซื้อเป็นหลัก

---

## 2. การยกเลิกตามสถานะคำสั่งซื้อ

### 2.1 สถานะ "รอชำระเงิน" (Pending Payment)

**ยกเลิกได้ทันที**

คำสั่งซื้อที่อยู่ในสถานะรอชำระเงินสามารถยกเลิกได้ทันทีโดยไม่มีค่าใช้จ่าย ผ่านแอปพลิ


### 1.2 Inject hierarchy into MD files

In [ ]:
import re
from pathlib import Path

def inject_hierarchy(text):
    """
    แก้ไข markdown ให้ทุก subheader มี parent path กำกับ
    ## foo         → ## foo (H1)
    ### bar        → ### bar (H1 > foo)
    """
    lines = text.split("\n")
    result = []
    h1, h2 = "", ""

    for line in lines:
        # จับ h1
        if re.match(r'^# (?!#)', line):
            h1 = line.lstrip("# ").strip()
            h2 = ""
            result.append(line)

        # แก้ h2: เติม (h1) ต่อท้าย
        elif re.match(r'^## (?!#)', line):
            h2 = line.lstrip("# ").strip()
            # กันไม่ให้เติมซ้ำถ้ารันสองรอบ
            if f"({h1})" not in line:
                result.append(f"{line} ({h1})")
            else:
                result.append(line)

        # แก้ h3: เติม (h1 > h2) ต่อท้าย
        elif re.match(r'^### (?!#)', line):
            h3 = line.lstrip("# ").strip()
            parent = f"{h1} > {h2}" if h2 else h1
            if f"({parent})" not in line:
                result.append(f"{line} ({parent})")
            else:
                result.append(line)

        else:
            result.append(line)

    return "\n".join(result)


# --- รันกับทุกไฟล์ใน knowledge base ---

kb_dir = Path(KB_DIR)
all_files = sorted(kb_dir.rglob("*.md"))
print(f"Found {len(all_files)} files\n")

PREVIEW_FILES = 2   # จำนวนไฟล์ที่จะแสดง preview
PREVIEW_LINES = 30  # จำนวนบรรทัดต่อไฟล์

for i, fp in enumerate(all_files):
    original = fp.read_text(encoding="utf-8")
    modified = inject_hierarchy(original)
    fp.write_text(modified, encoding="utf-8")

    # แสดง preview ไฟล์แรก ๆ
    if i < PREVIEW_FILES:
        orig_lines = original.split("\n")
        mod_lines  = modified.split("\n")

        print(f"{'='*60}")
        print(f"FILE: {fp.relative_to(kb_dir)}")
        print(f"{'='*60}")

        # แสดงเฉพาะบรรทัดที่เปลี่ยน
        changes = 0
        for j, (o, m) in enumerate(zip(orig_lines, mod_lines)):
            if o != m:
                print(f"  line {j+1}")
                print(f"    BEFORE: {o}")
                print(f"    AFTER:  {m}")
                changes += 1
        print(f"\n  → {changes} lines changed")

        # แสดง PREVIEW_LINES บรรทัดแรกของไฟล์หลังแก้
        print(f"\n--- First {PREVIEW_LINES} lines after ---")
        for line in mod_lines[:PREVIEW_LINES]:
            print(f"  {line}")
        print()

print(f"{'='*60}")
print(f"Done! {len(all_files)} files updated")
print(f"\n⚠️  reload documents ด้านล่างก่อนรัน chunking")

Found 118 files

FILE: policies/cancellation_policy.md
  line 7
    BEFORE: ## 1. ภาพรวมนโยบาย
    AFTER:  ## 1. ภาพรวมนโยบาย (นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่)
  line 13
    BEFORE: ## 2. การยกเลิกตามสถานะคำสั่งซื้อ
    AFTER:  ## 2. การยกเลิกตามสถานะคำสั่งซื้อ (นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่)
  line 15
    BEFORE: ### 2.1 สถานะ "รอชำระเงิน" (Pending Payment)
    AFTER:  ### 2.1 สถานะ "รอชำระเงิน" (Pending Payment) (นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่ > 2. การยกเลิกตามสถานะคำสั่งซื้อ)
  line 25
    BEFORE: ### 2.2 สถานะ "ชำระเงินแล้ว — กำลังเตรียมจัดส่ง" (Processing / Preparing to Ship)
    AFTER:  ### 2.2 สถานะ "ชำระเงินแล้ว — กำลังเตรียมจัดส่ง" (Processing / Preparing to Ship) (นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่ > 2. การยกเลิกตามสถานะคำสั่งซื้อ)
  line 40
    BEFORE: ### 2.3 สถานะ "จัดส่งแล้ว" (Shipped / In Transit)
    AFTER:  ### 2.3 สถานะ "จัดส่งแล้ว" (Shipped / In Transit) (นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่ > 2. การยกเลิกตามสถานะคำสั่งซื้อ)
  line 

### 1.3 Chunking

LLMs have limited context windows, and long documents dilute relevance. We split each document into smaller **chunks** using a fixed-size sliding window with overlap.

In [ ]:
# วัดขนาด section จริงจากทุกไฟล์ใน knowledge base

section_sizes = []

for doc in documents:
    sections = re.split(r'\n(?=#{2,3} )', doc["text"])
    for section in sections:
        if section.strip():
            section_sizes.append({
                "source": doc["path"],
                "header": section.split("\n")[0][:60],
                "size": len(section)
            })

section_sizes.sort(key=lambda x: x["size"], reverse=True)

# สถิติรวม
sizes = [s["size"] for s in section_sizes]
print(f"จำนวน sections ทั้งหมด: {len(sizes)}")
print(f"เล็กสุด:  {min(sizes)} chars")
print(f"ใหญ่สุด:  {max(sizes)} chars")
print(f"เฉลี่ย:   {int(sum(sizes)/len(sizes))} chars")

# distribution
thresholds = [256, 512, 768, 1024, 1536, 2048]
print(f"\n--- % ของ sections ที่ไม่โดนตัดที่ขนาดต่าง ๆ ---")
for t in thresholds:
    pct = sum(1 for s in sizes if s <= t) / len(sizes) * 100
    print(f"  MAX_CHUNK={t:>5}: ครอบคลุม {pct:>5.1f}% ของ sections")

# top 10 section ที่ใหญ่สุด
print(f"\n--- Top 10 sections ที่ใหญ่สุด ---")
for s in section_sizes[:10]:
    print(f"  {s['size']:>5} chars | {s['source']:<40} | {s['header']}")

จำนวน sections ทั้งหมด: 885
เล็กสุด:  27 chars
ใหญ่สุด:  1929 chars
เฉลี่ย:   431 chars

--- % ของ sections ที่ไม่โดนตัดที่ขนาดต่าง ๆ ---
  MAX_CHUNK=  256: ครอบคลุม  39.5% ของ sections
  MAX_CHUNK=  512: ครอบคลุม  69.3% ของ sections
  MAX_CHUNK=  768: ครอบคลุม  85.1% ของ sections
  MAX_CHUNK= 1024: ครอบคลุม  95.1% ของ sections
  MAX_CHUNK= 1536: ครอบคลุม  99.4% ของ sections
  MAX_CHUNK= 2048: ครอบคลุม 100.0% ของ sections

--- Top 10 sections ที่ใหญ่สุด ---
   1929 chars | products/SF-SP-001_saifah_phone_x9_pro_max.md | ## รายละเอียดสินค้า
   1649 chars | products/SF-SP-001_saifah_phone_x9_pro_max.md | ## คำถามที่พบบ่อยเกี่ยวกับสินค้านี้
   1613 chars | products/DN-LT-006_daonuea_probook_14_max.md | ## รายละเอียดสินค้า
   1544 chars | products/DN-LT-004_daonuea_probook_16.md | ## รายละเอียดสินค้า
   1543 chars | products/SF-SP-011_saifah_phone_duopad.md | ## รายละเอียดสินค้า
   1501 chars | products/DN-LT-006_daonuea_probook_14_max.md | ## คำถามที่พบบ่อยเกี่ยวกับสินค้านี้
   1497 chars

In [ ]:
MAX_CHUNK = 1024         # ค่า default ครอบคลุม 95%
MAX_CHUNK_FAQ = 512      # FAQ ตัดเล็กกว่า เพราะแต่ละ Q&A ควรเป็น chunk ของตัวเอง
MAX_CHUNK_DETAIL = 1536  # รายละเอียดสินค้า ยอมให้ใหญ่หน่อย context สำคัญ
CHUNK_OVERLAP = 128

def make_chunks_by_section(text, source_path):
    sections = re.split(r'\n(?=#{2,3} )', text)
    result = []

    for section in sections:
        if not section.strip():
            continue

        first_line = section.split("\n")[0]

        # เลือก max size ตาม section type
        if "คำถามที่พบบ่อย" in first_line:
          # split ที่ --- แทน เพราะแต่ละ Q&A จบด้วย ---
          qa_chunks = re.split(r'\n---\n', section)
          for qa in qa_chunks:
            if qa.strip() and qa.strip() != "---":
              enriched = f"{first_line}\n{qa.strip()}"
              result.append({"text": enriched, "source": source_path})
          continue
        elif "รายละเอียดสินค้า" in first_line:
            max_size = MAX_CHUNK_DETAIL
        else:
            max_size = MAX_CHUNK

        # chunking ปกติ
        if len(section) <= max_size:
            result.append({"text": section.strip(), "source": source_path})
        else:
            start = 0
            while start < len(section):
                window = section[start:start + max_size]
                if start > 0:
                    window = f"{first_line} (ต่อ)\n{window.strip()}"
                result.append({"text": window.strip(), "source": source_path})
                start += max_size #- CHUNK_OVERLAP

    return result

# debug print
chunks = []
for doc in documents:
    chunks.extend(make_chunks_by_section(doc["text"], doc["path"]))

print(f"Created {len(chunks)} chunks from {len(documents)} documents\n")

# หา 1 ตัวอย่างของแต่ละ type
examples = {
    "FAQ": None,
    "รายละเอียดสินค้า": None,
    "ปกติ (ตัดพอดี)": None,
    "ปกติ (ต่อ)": None,
}

for c in chunks:
    first = c["text"].split("\n")[0]
    second = c["text"].split("\n")[1] if len(c["text"].split("\n")) > 1 else ""

    if examples["FAQ"] is None and "**Q:" in c["text"]:
        examples["FAQ"] = c
    if examples["รายละเอียดสินค้า"] is None and "รายละเอียดสินค้า" in first:
        examples["รายละเอียดสินค้า"] = c
    if examples["ปกติ (ต่อ)"] is None and "(ต่อ)" in first:
        examples["ปกติ (ต่อ)"] = c
    if examples["ปกติ (ตัดพอดี)"] is None and "**Q:" not in c["text"] \
        and "รายละเอียดสินค้า" not in first and "(ต่อ)" not in first:
        examples["ปกติ (ตัดพอดี)"] = c

    if all(v is not None for v in examples.values()):
        break

# แสดงผล
for label, c in examples.items():
    if c is None:
        print(f"{'='*60}")
        print(f"[{label}] — ไม่พบตัวอย่าง")
        continue
    print(f"{'='*60}")
    print(f"[{label}] source: {c['source']}")
    print(f"size: {len(c['text'])} chars")
    print(f"---")
    print(c["text"])
    print("...")

Created 901 chunks from 118 documents

[FAQ] source: products/JC-CS-006_judchuam_saifah_pen_gen2.md
size: 464 chars
---
## คำถามที่พบบ่อยเกี่ยวกับสินค้านี้
## คำถามที่พบบ่อยเกี่ยวกับสินค้านี้

**Q: SaiFah Pen Gen 2 ใช้กับ แท็บ S9 ได้ไหมครับ?**
A: ไม่ได้ครับ แท็บ S9 รองรับเฉพาะ SaiFah Pen Gen 1 เท่านั้น Gen 2 ออกแบบสำหรับ แท็บ S9 Pro และ แท็บ Mini 7 หากใช้ แท็บ S9 กรุณาหา Gen 1 (ปัจจุบันไม่มีจำหน่ายแยก — มักรวมมากับแพ็กเกจแท็บเล็ต)

**Q: ต้องชาร์จปากกายังไงครับ?**
A: แค่แปะปากกาเข้าด้านข้างของแท็บ S9 Pro หรือ แท็บ Mini 7 แม่เหล็กจะดูดติดและชาร์จได้ทันที ชาร์จเต็มในประมาณ 15 นาที
...
[รายละเอียดสินค้า] source: products/AW-MN-001_arcwave_proview_27_4k.md
size: 731 chars
---
## รายละเอียดสินค้า

ArcWave ProView 27 4K คือจอมอนิเตอร์ภายนอกขนาด 27 นิ้ว ความละเอียด 4K (3840×2160) พาเนล IPS ที่ออกแบบมาสำหรับงานที่ต้องการความแม่นยำของสีสูง ไม่ว่าจะเป็นการตกแต่งภาพ, ตัดต่อวิดีโอ, งานออกแบบกราฟิก หรือการทำงานทั่วไปที่ต้องการพื้นที่หน้าจอกว้างขึ้น ความครอบคลุม sRGB 99% ทำให้สีที่เห็นบนจอใกล้เคียงกั

### 1.3 Embedding

We use `paraphrase-multilingual-MiniLM-L12-v2` — a small (118M params), multilingual embedding model that produces 384-dimensional vectors. It supports Thai out of the box and doesn't require any special prefixes.

In [ ]:
from sentence_transformers import SentenceTransformer

# embed_model = SentenceTransformer("intfloat/multilingual-e5-large")
embed_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Embed all chunks
chunk_texts = [c["text"] for c in chunks]
chunk_embeddings = embed_model.encode(chunk_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

print(f"Chunk embeddings shape: {chunk_embeddings.shape}")  # (n_chunks, 384)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Chunk embeddings shape: (901, 384)


### 1.4 Retrieve

Embed the question, then find the most similar chunks via dot product (= cosine similarity for normalized vectors).

In [ ]:
TOP_K = 10

# ── 1. Keyword Classify (fallback สำหรับคำถามสั้น) ────────────
def classify_question(query: str) -> str:
    q = query.lower()
    compare_kw = ["เปรียบเทียบ", "ต่างกัน", "vs", "ดีกว่า", "แตกต่าง"]
    calc_kw    = ["รวม", "ทั้งหมด", "บวก", "ส่วนต่าง", "ต่างกันเท่าไหร่"]
    multi_kw   = ["และ", "กับ", "ทั้ง"]

    if any(k in q for k in compare_kw):
        return "compare"
    elif any(k in q for k in calc_kw):
        return "calculate"
    elif any(k in q for k in multi_kw):
        return "multi_product"
    else:
        return "simple"

# ── 2. Extract Intent + Classify + Decompose (1 LM call) ──────
INTENT_SYSTEM = """You are a query analyzer for a Thai electronics store RAG system.

Question types:
- simple: one product spec/price/availability, OR store policy (return, warranty, payment, shipping, trade-in, membership, order cancellation)
- compare: compares 2+ products on the same attribute
- multi_product: asks about 2+ products but not a direct comparison
- calculate: involves price math of STORE PRODUCTS only (not travel, food, or activities)
- out_of_scope: completely unrelated to products or store (e.g. weather, cooking, travel)
  *** Only use out_of_scope if there is NO product or store question at all ***

Rules for QUERY:
- Keep product names and policy topics — never remove them
- Policy topics include: คืนสินค้า, ยกเลิกออเดอร์, ประกัน, การชำระเงิน, จัดส่ง
- Remove personal context (stories, opinions)
- Translate casual Thai to product terms e.g. "แบตอึด"→แบตเตอรี่นาน, "ตัดเสียง"→ANC, "วัดหัวใจ"→ECG, "แตะจ่าย"→NFC Pay, "ตกไม่แตก"→กันกระแทก rubber bumper, "ดำน้ำได้"→IP67 IP68, "ฟัง Lossless"→LDAC, "จอสีแม่น"→DCI-P3, "เทิร์นเครื่อง"→Trade-in, "ใส่ซิม 2 ใบ"→Dual SIM, "ปุ่มฉุกเฉิน"→SOS Button, "ไม่มีพัดลม"→fanless, "จำกัดเวลาเด็ก"→Parental Control Kids Mode
- 5-15 words, Thai only

Rules for SUB_QUERIES:
- Fill only if TYPE is compare or multi_product
- One sub-query per product, separated by " | "
- Each sub-query must include product name + attribute being asked
- Leave blank for all other types

Output format (strictly follow this):
TYPE: <type>
QUERY: <clean Thai search query>
SUB_QUERIES: <sub-queries separated by " | ", or blank>"""

# product/policy signals สำหรับ double-check out_of_scope
PRODUCT_SIGNALS = [
    "สายฟ้า", "ดาวเหนือ", "วงโคจร", "คลื่นเสียง", "อาร์คเวฟ", "ฟ้าใหม่",
    "ประกัน", "คืนสินค้า", "ยกเลิก", "จัดส่ง", "care+",
    "watch", "airbook", "stormbook", "headon", "headpro",
    "บัดส์", "แท็บ", "หูฟัง", "มือถือ", "โน้ตบุ๊ค", "แล็ปท็อป",
]

def extract_intent(question: str) -> tuple[str, str, list[str]]:
    """Returns (q_type, clean_query, sub_queries) in one LM call.
    Fallback to keyword classify สำหรับคำถามสั้น"""
    if len(question) < 30:
        return classify_question(question), question, []

    # ── ตัดเอาแค่ท้าย 300 ตัวอักษรสำหรับข้อความยาว ──
    text_to_analyze = question[-300:] if len(question) > 300 else question

    # ── ลอง model หลักก่อน fallback ไป typhoon ──
    for model in ["kbtg", "typhoon"]:
        result = ask_llm([
            {"role": "system", "content": INTENT_SYSTEM},
            {"role": "user",   "content": text_to_analyze}
        ], model=model)

        if not result:
            continue

        # ── ตัด <think> block ออกก่อน parse ──
        clean_result = re.sub(r"<think>.*?</think>", "", result, flags=re.DOTALL).strip()

        q_type, clean_query, sub_queries = "simple", None, []
        for line in clean_result.splitlines():
            if line.startswith("TYPE:"):
                q_type = line.split(":", 1)[1].strip().lower()
            elif line.startswith("QUERY:"):
                clean_query = line.split(":", 1)[1].strip()
            elif line.startswith("SUB_QUERIES:"):
                raw = line.split(":", 1)[1].strip()
                sub_queries = [q.strip() for q in raw.split("|") if q.strip()]

        # ── validate: clean_query ต้องสั้นกว่า text_to_analyze ──
        if clean_query and len(clean_query) < len(text_to_analyze) * 0.8:
            # ── double-check out_of_scope ด้วย product signals จาก original question ──
            if q_type == "out_of_scope":
                has_signal = any(s in question.lower() for s in PRODUCT_SIGNALS)
                if has_signal:
                    print(f"[WARN] out_of_scope overridden — product signal found")
                    q_type = "simple"
            return q_type, clean_query, sub_queries

        print(f"[WARN] {model} parse failed, trying next model...")

    # ── fallback สุดท้าย: keyword classify + ตัดท้าย 50 ตัวอักษร ──
    print("[WARN] All models failed, using keyword fallback")
    return classify_question(text_to_analyze), text_to_analyze[-50:], []

# ── 3. Retrieve ───────────────────────────────────────────────
def dense_retrieve(query, chunk_embs, k=TOP_K):
    """Return (top_idx, scores_out, q_type, clean_query).
    Returns empty arrays immediately if out_of_scope."""

    # Step 1: extract intent + classify + decompose ใน 1 LM call
    q_type, retrieval_query, sub_queries = extract_intent(query)

    print(f"[Original   ]: {query[:60]}...")
    print(f"[Clean query]: {retrieval_query}")
    print(f"[Type       ]: {q_type}")

    # Step 2: short-circuit ถ้า out_of_scope — ไม่ต้อง retrieve เลย
    if q_type == "out_of_scope":
        return [], np.array([]), q_type, retrieval_query

    # Step 3: เลือก retrieve strategy ตาม type
    if q_type in ("compare", "multi_product"):
        if not sub_queries:
            sub_queries = [retrieval_query]
        print(f"[Decomposed ]: {sub_queries}")

        all_idx = {}
        for sub_q in sub_queries:
            q_emb  = embed_model.encode([sub_q], normalize_embeddings=True)
            scores = np.dot(chunk_embs, q_emb.T).flatten()
            top_idx = np.argsort(scores)[::-1][:k]
            for idx in top_idx:
                if idx not in all_idx or scores[idx] > all_idx[idx]:
                    all_idx[idx] = scores[idx]

        sorted_items = sorted(all_idx.items(), key=lambda x: x[1], reverse=True)
        sorted_items = sorted_items[:k * len(sub_queries)]
        top_idx    = [i for i, _ in sorted_items]
        scores_out = np.array([s for _, s in sorted_items])

    elif q_type == "calculate":
        q_emb  = embed_model.encode([retrieval_query], normalize_embeddings=True)
        scores = np.dot(chunk_embs, q_emb.T).flatten()
        top_idx    = np.argsort(scores)[::-1][:k * 2]
        scores_out = scores[top_idx]

    else:  # simple
        q_emb  = embed_model.encode([retrieval_query], normalize_embeddings=True)
        scores = np.dot(chunk_embs, q_emb.T).flatten()
        top_idx    = np.argsort(scores)[::-1][:k]
        scores_out = scores[top_idx]

    return top_idx, scores_out, q_type, retrieval_query

# ── Demo ──────────────────────────────────────────────────────
q = questions[8]
idx, scores, q_type, clean_query = dense_retrieve(q["question"], chunk_embeddings)

print(f"Question: {q['question']}\n")

if q_type == "out_of_scope":
    print("ANSWER: 10")
else:
    for rank, (i, s) in enumerate(zip(idx, scores), 1):
        print(f"  Rank {rank} (score={s:.3f}) [{chunks[i]['source']}]")
        print(f"  {chunks[i]['text']}...")
        print()

[Original   ]: เฮ้ยย อ่านเยอะมากเลย ขอถามนิดนึงนะคะ คือพอดีว่าจะไปเที่ยวภูเ...
[Clean query]: สายฟ้า Rugged R1 กันน้ำระดับไหน ประกันคุ้มครองน้ำเค็มมั้ย
[Type       ]: simple
Question: เฮ้ยย อ่านเยอะมากเลย ขอถามนิดนึงนะคะ คือพอดีว่าจะไปเที่ยวภูเก็ตกับเพื่อนสนิท 3 คนช่วงวันหยุดยาวเดือนหน้า จองโรงแรม Kata Beach Resort ไว้แล้ว 3 คืน เพื่อนบอกว่าตรงนั้นมีจุดดำน้ำดูปะการังสวยมากชื่อ Freedom Beach ต้องนั่งเรือหางยาวไป ค่าเรือคนละ 800 บาท แล้วก็มีร้านอาหารทะเลอร่อยชื่อ No.6 Restaurant ที่ป่าตอง เพื่อนแนะนำให้ลองกุ้งมังกรเผา แล้วก็พอดีว่าเราชอบถ่ายรูปใต้น้ำด้วย เพื่อนอีกคนบอกว่าเคยเอามือถือกันน้ำไปถ่าย ได้ภาพสวยมาก เลยอยากรู้ว่าสายฟ้า Rugged R1 กันน้ำระดับไหนคะ เอาไปดำน้ำถ่ายรูปใต้ทะเลได้มั้ย แล้วถ้าน้ำเค็มเข้าเครื่องประกันคุ้มครองมั้ย

  Rank 1 (score=0.689) [products/SF-SP-015_saifah_phone_rugged_r1.md]
  ## รายละเอียดสินค้า

สายฟ้า โฟน Rugged R1 สร้างขึ้นสำหรับงานหนักและสภาพแวดล้อมที่ท้าทาย ตัวเครื่องผ่านมาตรฐานทหาร MIL-STD-810H ครอบคลุมการทดสอบ 14 รายการ ได้แก่ การตก อุณหภูมิสุดขั้ว ความชื้

### 1.5 Generate Answer

Send the retrieved context + question + choices to the LLM and parse the answer.

In [ ]:
# A very basic system prompt — you should improve this!
SYSTEM_PROMPT = """You are a customer service assistant for FahMai electronics store.
Answer questions using ONLY the provided reference information. Do not guess or use outside knowledge.

## How to select an answer
- If the reference information clearly answers the question → choose the correct option
- If the question is about FahMai but the information is NOT in the reference → choose option 9
- If the question is completely unrelated to FahMai products or store → choose option 10

## When to choose option 9
Only choose 9 after reading ALL reference chunks and none of them answer the question.
Do NOT choose 9 just because one chunk seems incomplete — another chunk may have the answer.

## When to choose option 10
The question has absolutely no connection to FahMai products, pricing, policies, or store services.
This includes: recipes, flights, bank interest rates, weather, government holidays, calendars,
general knowledge, news, sports, entertainment, or anything not related to electronics retail.
*** If the question would make no sense to ask an electronics store, choose option 10 ***

## Reading the reference
- Read ALL chunks completely before deciding — information may be spread across multiple chunks
- If chunks contain seemingly conflicting information, use the chunk most relevant to the question
- For long questions with personal stories, focus only on the product or store question at the end

## Special cases
- Calculation questions: show the calculation steps using reference data, then choose the answer
- Comparison questions: gather information from all relevant chunks before comparing
- Multi-condition questions: verify that the answer satisfies ALL conditions in the question

You may reason step by step, but always end with ANSWER: X"""

def get_source_label(source_path):
    if "products/" in source_path:
        return "ข้อมูลสินค้า"
    elif "policies/" in source_path:
        return "นโยบายร้าน"
    elif "store_info/" in source_path:
        return "ข้อมูลร้าน"
    return source_path

def build_rag_prompt(clean_query, choices, retrieved_chunks):
    context = "\n\n".join(
        f"[ข้อมูล {i+1} | {get_source_label(c['source'])}]\n{c['text']}"
        for i, c in enumerate(retrieved_chunks)
    )
    choices_text = "\n".join(f"{k}. {v}" for k, v in choices.items())

    calc_keywords = ["รวม", "ราคารวม", "เท่าไหร่", "กี่ points",
                     "ลดได้", "จ่ายเพิ่ม", "ทั้งหมด", "คำนวณ"]
    calc_hint = "\n💡 คำถามนี้อาจต้องคำนวณ ให้แสดงขั้นตอนก่อนแล้วค่อยตอบ ANSWER\n" \
                if any(kw in clean_query for kw in calc_keywords) else ""

    return (
        f"ข้อมูลอ้างอิง:\n{context}\n\n"
        f"โปรดอ่านข้อมูลด้านบนแล้วตอบคำถามต่อไปนี้\n"
        f"{calc_hint}\n"
        f"คำถาม: {clean_query}\n\n"
        f"ตัวเลือก:\n{choices_text}\n\n"
        f"หมายเหตุ: ตัวเลือก 9 = ไม่มีข้อมูลนี้ในฐานข้อมูล, "
        f"ตัวเลือก 10 = คำถามไม่เกี่ยวกับร้านฟ้าใหม่\n\n"
        f"ANSWER:"
    )

# Demo: answer Q1
q = questions[99]
idx, scores, q_type, clean_query = dense_retrieve(q["question"], chunk_embeddings)

print(f"Q{q['id']}: {q['question']}")

if q_type == "out_of_scope":
    print("ANSWER: 10")
else:
    retrieved = [chunks[i] for i in idx]
    prompt = build_rag_prompt(clean_query, q["choices"], retrieved)
    raw = ask_llm([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ])
    answer = parse_answer(raw)
    print(f"LLM raw: {raw}")
    print(f"Parsed answer: {answer}")

[Original   ]: สตอร์มบุ๊ก G5 ราคา ฿27,990 ซื้อมาได้ 5 วัน ยังไม่แกะกล่อง อย...
[Clean query]: สตอร์มบุ๊ก G5 คืนสินค้าได้ไหม
[Type       ]: simple
Q100: สตอร์มบุ๊ก G5 ราคา ฿27,990 ซื้อมาได้ 5 วัน ยังไม่แกะกล่อง อยากคืนสินค้าได้ไหมครับ
LLM raw: <think>
Okay, let's tackle this question. The user is asking whether the StormBook G5 can be returned. I need to check the provided reference information to find the answer.

First, I'll look through the data chunks. In [ข้อมูล 3], there's a note about clearance items. It says that StormBook G5 (2024) is a clearance item and cannot be returned or exchanged under any circumstances, except for manufacturing defects, which are considered on a case-by-case basis. 

Then, in [ข้อมูล 7], there's a promotion for a previous generation model, which is not a clearance item and can be returned according to normal policies. However, the question is about the StormBook G5 (2024), which is specifically mentioned as a clearance item in [ข้อมูล 3].

Looking at th

In [ ]:
retrieved

[{'text': '# ดาวเหนือ สตอร์มบุ๊ก G5 (2024) (DaoNuea StormBook G5 (2024))\n\nรหัสสินค้า: DN-LT-009\nแบรนด์: ดาวเหนือ (DaoNuea) — แบรนด์ในเครือฟ้าใหม่\nหมวดหมู่: แล็ปท็อปเกมมิ่ง\nราคา: ฿27,990\nสถานะ: สินค้าลดล้างสต็อก (CLEARANCE)\nวันที่อัปเดต: 1 มีนาคม 2569\n\n---',
  'source': 'products/DN-LT-009_daonuea_stormbook_g5_2024.md'},
 {'text': '# ดาวเหนือ สตอร์มบุ๊ก G5 (DaoNuea StormBook G5)\n\nรหัสสินค้า: DN-LT-008\nแบรนด์: ดาวเหนือ (DaoNuea) — แบรนด์ในเครือฟ้าใหม่\nหมวดหมู่: แล็ปท็อปเกมมิ่ง\nราคา: ฿32,990\nสถานะ: มีสินค้า\nวันที่อัปเดต: 1 มีนาคม 2569\n\n---',
  'source': 'products/DN-LT-008_daonuea_stormbook_g5.md'},
 {'text': '## หมายเหตุพิเศษ\n\n> **สินค้าลดล้างสต็อก — ไม่สามารถคืนหรือเปลี่ยนได้**\n>\n> StormBook G5 (2024) เป็นสินค้า Clearance ที่ลดราคาเพื่อระบายสต็อก สินค้านี้ **ไม่รับคืนและไม่เปลี่ยน** ในทุกกรณี (ยกเว้นสินค้ามีตำหนิจากการผลิต ซึ่งจะพิจารณาเป็นรายกรณี) กรุณาตรวจสอบสเปคและสภาพสินค้าให้ละเอียดก่อนยืนยันคำสั่งซื้อ\n\n---',
  'source': 'products/DN-LT-009_daonuea_stormbook

### 1.6 Run All Questions (Dense)

Loop through questions, retrieve, generate, and collect predictions.

In [ ]:
def run_pipeline(questions, retrieve_fn, label="pipeline", n=N_QUESTIONS):
    """Run retrieval + LLM on first n questions. Returns predictions dict.
    retrieve_fn(query) must return (idx_list, clean_query, q_type)
    """
    predictions = {}
    for i, q in enumerate(questions[:n]):
        idx_list, clean_query, q_type = retrieve_fn(q["question"])

        if q_type == "out_of_scope":
            pred = 10
            print(f"  Q{q['id']:>3}: out_of_scope → pred=10")
        else:
            retrieved_chunks = [chunks[i] for i in idx_list]
            prompt = build_rag_prompt(clean_query, q["choices"], retrieved_chunks)
            raw = ask_llm([
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ])
            pred = parse_answer(raw)
            if not pred:
                pred = 1  # default ถ้า parse fail
            print(f"  Q{q['id']:>3}: pred={pred}")

        predictions[q["id"]] = pred
        time.sleep(0.3)  # be nice to the API

    print(f"\n{label}: answered {len(predictions)} questions")
    return predictions


# ── Dense wrapper ──────────────────────────────────────────────
def dense_retrieve_fn(query):
    """Wrapper for run_pipeline: returns (idx_list, clean_query, q_type)"""
    idx, scores, q_type, clean_query = dense_retrieve(query, chunk_embeddings)
    return idx, clean_query, q_type

dense_preds = run_pipeline(questions, dense_retrieve_fn, label="dense")


[WARN] kbtg parse failed, trying next model...
[WARN] typhoon parse failed, trying next model...
[WARN] All models failed, using keyword fallback
[Original   ]: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ...
[Clean query]: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ
[Type       ]: simple
  Q  1: pred=5
[Original   ]: ปากกา SaiFah Pen Gen 2 ราคาเท่าไหร่คะ...
[Clean query]: ราคาปากกา SaiFah Pen Gen 2
[Type       ]: simple
  Q  2: pred=7
  Error: 502 Server Error: Bad Gateway for url: http://thaillm.or.th/api/kbtg/v1/chat/completions, retrying in 1s...
[WARN] kbtg parse failed, trying next model...
[WARN] typhoon parse failed, trying next model...
[WARN] All models failed, using keyword fallback
[Original   ]: หูฟัง HeadPro X1 ใช้ Bluetooth เวอร์ชันอะไรคะ...
[Clean query]: หูฟัง HeadPro X1 ใช้ Bluetooth เวอร์ชันอะไรคะ
[Type       ]: simple
  Q  3: pred=2
[Original   ]: อยากเอามือถือเครื่องเก่ามาเทิร์น เปลี่ยนเป็นเครื่องใหม่ที่ฟ้...
[Clean query]: ฟ้าใหม่ คืนเครื่องเก่า แลกเครื่องใหม่ได้ไหม
[Type       

KeyboardInterrupt: 

---
## Section 2: Sparse Retrieval (BM25)

**BM25** is a keyword-matching algorithm. It scores documents by how many query terms they contain, weighted by term rarity (IDF). No neural network needed.

### 2.1 Thai Tokenization

BM25 needs tokenized text. Thai has no spaces between words, so we use `pythainlp` to segment.

In [ ]:
from pythainlp.tokenize import word_tokenize

# Demo: tokenize a Thai sentence
sample = "Watch S3 Ultra กันน้ำได้กี่ ATM ครับ?"
tokens = word_tokenize(sample, engine="newmm")
print(f"Input:  {sample}")
print(f"Tokens: {tokens}")

Input:  Watch S3 Ultra กันน้ำได้กี่ ATM ครับ?
Tokens: ['Watch', ' ', 'S', '3', ' ', 'Ultra', ' ', 'กันน้ำ', 'ได้', 'กี่', ' ', 'ATM', ' ', 'ครับ', '?']


### 2.2 Build BM25 Index

In [ ]:
from rank_bm25 import BM25Okapi

# Tokenize all chunks
tokenized_chunks = [word_tokenize(c["text"], engine="newmm") for c in chunks]
bm25 = BM25Okapi(tokenized_chunks)

print(f"BM25 index built over {len(tokenized_chunks)} chunks")

BM25 index built over 901 chunks


### 2.3 Retrieve with BM25

Compare BM25 results with dense results for the same question.

In [ ]:
def bm25_retrieve(query, k=TOP_K):
    """Return top-k chunk indices using BM25."""
    tokens = word_tokenize(query, engine="newmm")
    scores = bm25.get_scores(tokens)
    top_idx = np.argsort(scores)[::-1][:k]
    return top_idx, scores[top_idx]

# # Demo: compare dense vs BM25
# q = questions[0]
# print(f"Question: {q['question']}\n")

# d_idx, _, q_type, clean_query = dense_retrieve(q["question"], chunk_embeddings)
# b_idx, _ = bm25_retrieve(clean_query)

# print(f"Clean query: {clean_query}")
# print("\nDense top-5 sources:")
# for i in d_idx[:5]:
#     print(f"  {chunks[i]['source']}")

# print("\nBM25 top-5 sources:")
# for i in b_idx[:5]:
#     print(f"  {chunks[i]['source']}")


### 2.4 Run All Questions (BM25)

In [ ]:
def bm25_retrieve_fn(query):
    """Wrapper for run_pipeline: returns (idx_list, clean_query, q_type)"""
    # BM25 ไม่มี intent extraction — ใช้ extract_intent แยกก่อน
    q_type, clean_query, _ = extract_intent(query)
    idx, _ = bm25_retrieve(clean_query)  # ใช้ clean_query แทน raw
    return list(idx), clean_query, q_type

bm25_preds = run_pipeline(questions, bm25_retrieve_fn, label="bm25")

---
## Section 3: Hybrid Retrieval (RRF)

**Hybrid** combines dense and sparse results. The idea: dense is good at semantic matching, BM25 is good at exact keyword matching. Together they cover more cases.

We use **Reciprocal Rank Fusion (RRF)** to merge the two ranked lists:

$$\text{score}(d) = \sum_{r \in \text{rankers}} \frac{1}{k + \text{rank}_r(d)}$$

where $k$ is a constant (typically 60). Documents ranked highly by *either* method get a high combined score.

In [ ]:
def hybrid_retrieve(query, chunk_embs, k=TOP_K, rrf_k=60):
    """Combine dense + BM25 results using Reciprocal Rank Fusion."""
    fetch_k = k * 2

    # dense_retrieve ตอนนี้ return 4 ค่า — unpack ให้ครบ
    d_idx, _, q_type, clean_query = dense_retrieve(query, chunk_embs, k=fetch_k)

    # ถ้า out_of_scope ไม่ต้อง fuse เลย
    if q_type == "out_of_scope":
        return [], clean_query, q_type

    b_idx, _ = bm25_retrieve(clean_query, k=fetch_k)  # ใช้ clean_query แทน raw query

    # Compute RRF scores
    rrf_scores = {}
    for rank, idx in enumerate(d_idx, 1):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (rrf_k + rank)
    for rank, idx in enumerate(b_idx, 1):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (rrf_k + rank)

    sorted_idx = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)[:k]
    return sorted_idx, clean_query, q_type

# Demo
# q = questions[0]
# h_idx, clean_query, q_type = hybrid_retrieve(q["question"], chunk_embeddings)
# print(f"Question: {q['question']}\n")
# print(f"Type: {q_type} | Clean query: {clean_query}")
# print("Hybrid top sources:")
# for i in h_idx[:5]:
#     print(f"  {chunks[i]['source']}")


[WARN] kbtg parse failed, trying next model...
[WARN] typhoon parse failed, trying next model...
[WARN] All models failed, using keyword fallback
[Original   ]: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ...
[Clean query]: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ
[Type       ]: simple
Question: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ

Type: simple | Clean query: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ
Hybrid top sources:
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  products/WK-SW-008_wongkhojon_watch_s2_pro.md
  products/WK-SW-003_wongkhojon_watch_s3.md
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  products/WK-SW-003_wongkhojon_watch_s3.md


### 3.2 Run All Questions (Hybrid)

In [ ]:
def hybrid_retrieve_fn(query):
    """Wrapper for run_pipeline: returns (idx_list, clean_query, q_type)"""
    idx, clean_query, q_type = hybrid_retrieve(query, chunk_embeddings)
    return idx, clean_query, q_type

hybrid_preds = run_pipeline(questions, hybrid_retrieve_fn, label="hybrid")


  Error: 502 Server Error: Bad Gateway for url: http://thaillm.or.th/api/kbtg/v1/chat/completions, retrying in 1s...
  Error: 504 Server Error: Gateway Timeout for url: http://thaillm.or.th/api/kbtg/v1/chat/completions, retrying in 2s...
[WARN] kbtg parse failed, trying next model...
[WARN] typhoon parse failed, trying next model...
[WARN] All models failed, using keyword fallback
[Original   ]: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ...
[Clean query]: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ
[Type       ]: simple
  Q  1: pred=5
[Original   ]: ปากกา SaiFah Pen Gen 2 ราคาเท่าไหร่คะ...
[Clean query]: ราคาปากกา SaiFah Pen Gen 2
[Type       ]: simple
  Q  2: pred=7
[WARN] kbtg parse failed, trying next model...
[WARN] typhoon parse failed, trying next model...
[WARN] All models failed, using keyword fallback
[Original   ]: หูฟัง HeadPro X1 ใช้ Bluetooth เวอร์ชันอะไรคะ...
[Clean query]: หูฟัง HeadPro X1 ใช้ Bluetooth เวอร์ชันอะไรคะ
[Type       ]: simple
  Q  3: pred=2
[Original   ]: อยากเอามือถือเ

### 3.3 Compare All Three Methods

In [ ]:
print(f"{'QID':>4}  {'Dense':>6} {'BM25':>6} {'Hybrid':>7}")
print("-" * 30)
for q in questions[:N_QUESTIONS]:
    qid = q["id"]
    d = dense_preds.get(qid, "-")
    b = bm25_preds.get(qid, "-")
    h = hybrid_preds.get(qid, "-")
    match = "" if d == b == h else "  ← differ"
    print(f"Q{qid:>3}  {d:>6} {b:>6} {h:>7}{match}")

### Write Submission

Pick your best method and generate a `submission.csv` for Kaggle.

> Set `N_QUESTIONS = 100` at the top and re-run the notebook to generate a full submission.

In [ ]:
# Use dense predictions as the submission (change to hybrid_preds or bm25_preds to try others)
best_preds = hybrid_preds

with open("submission.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "answer"])
    for q in questions:
        qid = q["id"]
        writer.writerow([qid, best_preds.get(qid, 1)])  # default=1 for unanswered

print(f"submission.csv written ({len(questions)} rows)")

submission.csv written (100 rows)
